# 04b: Build a previous-week weather panel for leakage-safe HPAI prediction

## 目的

予測対象週 \(t\) のHPAIリスクを評価する際に，同じ週の実測気象ではなく，
**直前に完了した週 \(t-1\) の実測気象**を説明変数として使用するためのパネルを作成する．

\[
\text{risk in week } t
\leftarrow
\text{weather observed in week } t-1
\]

本ノートブックは，既存ファイルを上書きせず，次のファイルを新規保存する．

### 主な出力

- `weather_weekly_previous_week_features.parquet`
- `hpai_weekly_grid_panel_with_previous_week_weather.parquet`
- `hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet`

### 前週気象列

- `temp_mean_c_lag1w`
- `temp_min_c_lag1w`
- `temp_max_c_lag1w`

`weather_source_week_start`は，その特徴量が実際に観測された週の月曜日を示す．
予測対象の`week_start`との差は必ず7日でなければならない．

## 重要

- 元の`hpai_weekly_grid_panel.parquet`および`weather_weekly_by_grid.parquet`は上書きしない．
- 現在週の気象列は最終パネルから除外する．
- 全行を保持したパネルと，前週気象がすべて存在するモデル用パネルを別々に保存する．
- 欠測率，重複，週のずれ，気温の大小関係，発生行の残存数を自動検査する．

## 1. Google Driveの接続と基本設定

既存プロジェクトと同じディレクトリ構造を使用する．
必要に応じて`BASE_DIR`だけ変更すること．

In [ ]:
# Google ColabではDriveを接続する
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab以外の環境として実行します．")
except Exception as e:
    print("Drive mount warning:", repr(e))

import json
import os
import platform
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# ============================================================
# 設定
# ============================================================
BASE_DIR = Path("/content/drive/MyDrive/avian_influenza_project")
PROC_DIR = BASE_DIR / "processed"
AUDIT_DIR = PROC_DIR / "04b_previous_week_weather_audit"

PROC_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# 既存の出力がある場合に上書きするか
OVERWRITE_OUTPUTS = True

# 巨大な全件CSVは通常保存しない．Parquetと先頭サンプルCSVを保存する．
SAVE_FULL_CSV = False
SAMPLE_N = 10000

# ============================================================
# 解析対象期間
# ============================================================
# 発生記録と既存原稿の確定期間に合わせる．
# 前週気象が存在しても，この終了日より後の週を陰性例として使用しない．
ANALYSIS_START = pd.Timestamp("2020-08-31")
ANALYSIS_END = pd.Timestamp("2026-05-18")

# 入力候補．先頭から優先的に採用する．
PANEL_CANDIDATES = [
    PROC_DIR / "hpai_weekly_grid_panel.parquet",
    PROC_DIR / "hpai_weekly_grid_panel_with_weather.parquet",
    PROC_DIR / "hpai_weekly_grid_panel_for_model.parquet",
]

WEATHER_CANDIDATES = [
    PROC_DIR / "weather_weekly_by_grid.parquet",
]

CURRENT_WEATHER_COLS = [
    "temp_mean_c",
    "temp_min_c",
    "temp_max_c",
]

LAG_WEATHER_RENAME = {
    "temp_mean_c": "temp_mean_c_lag1w",
    "temp_min_c": "temp_min_c_lag1w",
    "temp_max_c": "temp_max_c_lag1w",
}
LAG_WEATHER_COLS = list(LAG_WEATHER_RENAME.values())

print("BASE_DIR :", BASE_DIR)
print("PROC_DIR :", PROC_DIR)
print("AUDIT_DIR:", AUDIT_DIR)
print("Python   :", sys.version.split()[0])
print("pandas   :", pd.__version__)
print("platform :", platform.platform())
print("Analysis period:", ANALYSIS_START.date(), "to", ANALYSIS_END.date())

## 2. 補助関数

入力ファイル探索，安全な保存，列検査を行う．

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def require_columns(df, required, df_name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"{df_name} に必要な列がありません: {missing}\n"
            f"利用可能な列: {list(df.columns)}"
        )


def save_parquet(df, path, overwrite=True):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"出力済みファイルがあります: {path}\n"
            "上書きする場合はOVERWRITE_OUTPUTS=Trueにしてください．"
        )
    df.to_parquet(path, index=False)
    print(f"Saved: {path}  rows={len(df):,}  cols={df.shape[1]}")
    return path


def save_csv(df, path, overwrite=True):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"出力済みファイルがあります: {path}"
        )
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path}  rows={len(df):,}  cols={df.shape[1]}")
    return path


def file_info(path, role):
    path = Path(path)
    stat = path.stat()
    return {
        "role": role,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": int(stat.st_size),
        "modified_at": datetime.fromtimestamp(
            stat.st_mtime
        ).isoformat(timespec="seconds"),
    }


def detect_event_column(df):
    candidates = [
        "outbreak_binary",
        "outbreak_flag",
        "has_outbreak",
        "event",
        "y",
        "target",
    ]
    return next((c for c in candidates if c in df.columns), None)

## 3. 入力ファイルの探索

`hpai_weekly_grid_panel.parquet`と`weather_weekly_by_grid.parquet`を優先する．  
週次気象ファイルが見つからない場合に限り，現在週気象を含むパネルから週次気象表を復元する．

In [ ]:
panel_path = first_existing(PANEL_CANDIDATES)
weather_path = first_existing(WEATHER_CANDIDATES)

if panel_path is None:
    raise FileNotFoundError(
        "週次HPAIパネルが見つかりません．確認した候補:\n"
        + "\n".join(str(p) for p in PANEL_CANDIDATES)
    )

print("Selected panel :", panel_path)
print("Selected weather:", weather_path)

panel = pd.read_parquet(panel_path)
require_columns(panel, ["grid_id", "week_start"], "panel")
panel["grid_id"] = panel["grid_id"].astype(str)
panel["week_start"] = pd.to_datetime(panel["week_start"]).dt.normalize()

if weather_path is not None:
    weather_weekly = pd.read_parquet(weather_path)
    weather_source_description = str(weather_path)
else:
    require_columns(
        panel,
        ["grid_id", "week_start"] + CURRENT_WEATHER_COLS,
        "fallback panel",
    )
    print(
        "WARNING: weather_weekly_by_grid.parquetが見つからないため，"
        "パネル内の現在週気象列から週次気象表を復元します．"
    )
    weather_weekly = panel[
        ["grid_id", "week_start"] + CURRENT_WEATHER_COLS
    ].copy()
    weather_source_description = (
        f"reconstructed from current-week weather columns in {panel_path}"
    )

require_columns(
    weather_weekly,
    ["grid_id", "week_start"] + CURRENT_WEATHER_COLS,
    "weather_weekly",
)
weather_weekly["grid_id"] = weather_weekly["grid_id"].astype(str)
weather_weekly["week_start"] = pd.to_datetime(
    weather_weekly["week_start"]
).dt.normalize()

print("\npanel shape :", panel.shape)
print("weather shape:", weather_weekly.shape)
print(
    "panel period  :",
    panel["week_start"].min(),
    "to",
    panel["week_start"].max(),
)
print(
    "weather period:",
    weather_weekly["week_start"].min(),
    "to",
    weather_weekly["week_start"].max(),
)
print("panel grids  :", panel["grid_id"].nunique())
print("weather grids:", weather_weekly["grid_id"].nunique())

input_log_rows = [file_info(panel_path, "input_panel")]
if weather_path is not None:
    input_log_rows.append(file_info(weather_path, "input_weather_weekly"))

input_log = pd.DataFrame(input_log_rows)
save_csv(
    input_log,
    AUDIT_DIR / "04b_input_file_log.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
display(input_log)

## 4. 入力データの整合性確認

- `grid_id × week_start`が一意であることを確認する．
- 週開始日が月曜日であることを確認する．
- 週次気象に重複がある場合は，元の週次集計規則に従って再集約する．

In [ ]:
panel_duplicate_n = int(
    panel.duplicated(["grid_id", "week_start"]).sum()
)
if panel_duplicate_n > 0:
    dup = panel[
        panel.duplicated(["grid_id", "week_start"], keep=False)
    ].sort_values(["grid_id", "week_start"])
    display(dup.head(100))
    raise ValueError(
        f"panelにgrid_id × week_startの重複が"
        f"{panel_duplicate_n:,}件あります．"
    )

weather_duplicate_n = int(
    weather_weekly.duplicated(["grid_id", "week_start"]).sum()
)

if weather_duplicate_n > 0:
    print(
        f"WARNING: weather_weeklyに重複が"
        f"{weather_duplicate_n:,}件あります．再集約します．"
    )
    weather_weekly = (
        weather_weekly[
            ["grid_id", "week_start"] + CURRENT_WEATHER_COLS
        ]
        .groupby(["grid_id", "week_start"], as_index=False)
        .agg(
            {
                "temp_mean_c": "mean",
                "temp_min_c": "min",
                "temp_max_c": "max",
            }
        )
        .sort_values(["grid_id", "week_start"])
        .reset_index(drop=True)
    )

panel_non_monday = int((panel["week_start"].dt.weekday != 0).sum())
weather_non_monday = int(
    (weather_weekly["week_start"].dt.weekday != 0).sum()
)

if panel_non_monday > 0:
    raise ValueError(
        f"panelに月曜日以外のweek_startが"
        f"{panel_non_monday:,}行あります．"
    )
if weather_non_monday > 0:
    raise ValueError(
        f"weather_weeklyに月曜日以外のweek_startが"
        f"{weather_non_monday:,}行あります．"
    )

numeric_weather = weather_weekly[CURRENT_WEATHER_COLS].apply(
    pd.to_numeric, errors="coerce"
)
weather_weekly[CURRENT_WEATHER_COLS] = numeric_weather

input_validation = pd.DataFrame(
    [
        {
            "check": "panel duplicate grid-week",
            "value": panel_duplicate_n,
            "passed": panel_duplicate_n == 0,
        },
        {
            "check": "weather duplicate grid-week before repair",
            "value": weather_duplicate_n,
            "passed": True,
        },
        {
            "check": "panel non-Monday week_start",
            "value": panel_non_monday,
            "passed": panel_non_monday == 0,
        },
        {
            "check": "weather non-Monday week_start",
            "value": weather_non_monday,
            "passed": weather_non_monday == 0,
        },
    ]
)
save_csv(
    input_validation,
    AUDIT_DIR / "04b_input_validation.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
display(input_validation)

print("Weather missing ratios:")
display(weather_weekly[CURRENT_WEATHER_COLS].isna().mean())
print("Weather descriptive statistics:")
display(weather_weekly[CURRENT_WEATHER_COLS].describe())

## 5. 前週気象特徴量の作成

元の気象週を`weather_source_week_start`として保持し，  
その値を使用する予測対象週を7日後の`week_start`として設定する．

In [ ]:
previous_weather = weather_weekly[
    ["grid_id", "week_start"] + CURRENT_WEATHER_COLS
].copy()

previous_weather = previous_weather.rename(
    columns={"week_start": "weather_source_week_start"}
)

previous_weather["week_start"] = (
    previous_weather["weather_source_week_start"]
    + pd.Timedelta(days=7)
)

previous_weather = previous_weather.rename(
    columns=LAG_WEATHER_RENAME
)

previous_weather = previous_weather[
    [
        "grid_id",
        "week_start",
        "weather_source_week_start",
    ]
    + LAG_WEATHER_COLS
].sort_values(["grid_id", "week_start"]).reset_index(drop=True)

lag_duplicate_n = int(
    previous_weather.duplicated(["grid_id", "week_start"]).sum()
)
if lag_duplicate_n > 0:
    raise ValueError(
        f"前週気象特徴量にgrid_id × week_startの重複が"
        f"{lag_duplicate_n:,}件あります．"
    )

gap_days = (
    previous_weather["week_start"]
    - previous_weather["weather_source_week_start"]
).dt.days

if not gap_days.eq(7).all():
    display(previous_weather.loc[gap_days.ne(7)].head(100))
    raise AssertionError(
        "予測対象週と気象観測週の差が7日でない行があります．"
    )

print("previous_weather:", previous_weather.shape)
print(
    "prediction weeks:",
    previous_weather["week_start"].min(),
    "to",
    previous_weather["week_start"].max(),
)
print(
    "source weeks    :",
    previous_weather["weather_source_week_start"].min(),
    "to",
    previous_weather["weather_source_week_start"].max(),
)
print("week gap counts:")
display(gap_days.value_counts().sort_index().rename("rows").to_frame())
display(previous_weather.head())

## 6. HPAI週次パネルへの結合

再実行時の混入を防ぐため，元パネルに現在週気象列または既存の前週気象列が含まれていても，
一度削除してから新しい前週気象を結合する．

In [ ]:
columns_to_remove_before_merge = (
    CURRENT_WEATHER_COLS
    + LAG_WEATHER_COLS
    + ["weather_source_week_start"]
)

existing_removed_cols = [
    c for c in columns_to_remove_before_merge if c in panel.columns
]
panel_base = panel.drop(
    columns=columns_to_remove_before_merge,
    errors="ignore",
).copy()

print("Columns removed from input panel:", existing_removed_cols)

panel_previous_weather = panel_base.merge(
    previous_weather,
    on=["grid_id", "week_start"],
    how="left",
    validate="one_to_one",
)

if len(panel_previous_weather) != len(panel):
    raise AssertionError(
        "結合後の行数が元パネルと一致しません．"
    )

merged_duplicate_n = int(
    panel_previous_weather.duplicated(
        ["grid_id", "week_start"]
    ).sum()
)
if merged_duplicate_n > 0:
    raise AssertionError(
        f"結合後にgrid_id × week_startの重複が"
        f"{merged_duplicate_n:,}件あります．"
    )

matched = panel_previous_weather[
    "weather_source_week_start"
].notna()

merged_gap_days = (
    panel_previous_weather.loc[matched, "week_start"]
    - panel_previous_weather.loc[
        matched, "weather_source_week_start"
    ]
).dt.days

if not merged_gap_days.eq(7).all():
    raise AssertionError(
        "結合後データに7日以外の気象週差があります．"
    )

print("panel input :", panel.shape)
print("panel output:", panel_previous_weather.shape)
print("matched previous-week weather rows:", int(matched.sum()))
print("unmatched rows:", int((~matched).sum()))
print("duplicate grid-week:", merged_duplicate_n)
display(panel_previous_weather.head())

## 7. 値の対応関係を検証

ランダムサンプルについて，`temp_*_lag1w`が
`weather_source_week_start`の元気象値と完全に一致するか確認する．

In [ ]:
verification_source = weather_weekly[
    ["grid_id", "week_start"] + CURRENT_WEATHER_COLS
].rename(columns={"week_start": "weather_source_week_start"})

verification_sample = panel_previous_weather.loc[
    matched,
    [
        "grid_id",
        "week_start",
        "weather_source_week_start",
    ]
    + LAG_WEATHER_COLS,
].copy()

if len(verification_sample) > 50000:
    verification_sample = verification_sample.sample(
        n=50000,
        random_state=42,
    )

verification_sample = verification_sample.merge(
    verification_source,
    on=["grid_id", "weather_source_week_start"],
    how="left",
    validate="many_to_one",
)

value_check_rows = []
for current_col, lag_col in LAG_WEATHER_RENAME.items():
    a = verification_sample[lag_col]
    b = verification_sample[current_col]
    comparable = a.notna() & b.notna()
    max_abs_diff = (
        float((a[comparable] - b[comparable]).abs().max())
        if comparable.any()
        else np.nan
    )
    mismatch_n = int(
        (
            comparable
            & ~np.isclose(a, b, rtol=0.0, atol=1e-12)
        ).sum()
    )
    value_check_rows.append(
        {
            "lag_column": lag_col,
            "source_column": current_col,
            "sample_rows": len(verification_sample),
            "comparable_rows": int(comparable.sum()),
            "mismatch_rows": mismatch_n,
            "max_abs_difference": max_abs_diff,
            "passed": mismatch_n == 0,
        }
    )

value_check = pd.DataFrame(value_check_rows)
save_csv(
    value_check,
    AUDIT_DIR / "04b_previous_week_value_verification.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
display(value_check)

if not value_check["passed"].all():
    raise AssertionError(
        "前週気象値と元の気象値が一致しない行があります．"
    )

## 8. 欠測，気温順序，発生行残存数の診断

モデル用パネルは，解析確定期間内で，3つの前週気象列がすべて存在する行だけに限定する．  
元パネルの最初の週は，それ以前の気象週が存在しないため欠測になることがある．

解析終了日は`2026-05-18`に固定し，それ以後の未確定週を陰性例として使用しない．

In [ ]:
# 気温の論理的大小関係
complete_temp = panel_previous_weather[
    LAG_WEATHER_COLS
].notna().all(axis=1)

bad_temp_order = panel_previous_weather[
    complete_temp
    & ~(
        (
            panel_previous_weather["temp_min_c_lag1w"]
            <= panel_previous_weather["temp_mean_c_lag1w"]
        )
        & (
            panel_previous_weather["temp_mean_c_lag1w"]
            <= panel_previous_weather["temp_max_c_lag1w"]
        )
    )
].copy()

print("temperature-order errors:", len(bad_temp_order))
if len(bad_temp_order) > 0:
    display(
        bad_temp_order[
            [
                "grid_id",
                "week_start",
                "weather_source_week_start",
            ]
            + LAG_WEATHER_COLS
        ].head(100)
    )
    raise AssertionError(
        "temp_min <= temp_mean <= temp_maxを満たさない行があります．"
    )

# 発生記録の確定期間内かつ，前週気象の欠測がない行だけを
# モデル用パネルに採用する．
analysis_period_mask = panel_previous_weather["week_start"].between(
    ANALYSIS_START,
    ANALYSIS_END,
    inclusive="both",
)

model_ready = (
    panel_previous_weather.loc[analysis_period_mask]
    .dropna(subset=LAG_WEATHER_COLS)
    .copy()
)

outside_analysis_period_n = int((~analysis_period_mask).sum())
print("rows outside fixed analysis period:", f"{outside_analysis_period_n:,}")

event_col = detect_event_column(panel_previous_weather)
print("event column:", event_col)

if event_col is not None:
    event_all_n = int(
        pd.to_numeric(
            panel_previous_weather[event_col],
            errors="coerce",
        ).fillna(0).eq(1).sum()
    )
    event_model_n = int(
        pd.to_numeric(
            model_ready[event_col],
            errors="coerce",
        ).fillna(0).eq(1).sum()
    )
    event_retention = (
        event_model_n / event_all_n
        if event_all_n > 0
        else np.nan
    )
else:
    event_all_n = np.nan
    event_model_n = np.nan
    event_retention = np.nan

print("\nFull panel:")
print(" rows :", f"{len(panel_previous_weather):,}")
print(
    " weeks:",
    panel_previous_weather["week_start"].min(),
    "to",
    panel_previous_weather["week_start"].max(),
)
print(" grids:", panel_previous_weather["grid_id"].nunique())

print("\nModel-ready panel:")
print(" rows :", f"{len(model_ready):,}")
print(
    " weeks:",
    model_ready["week_start"].min(),
    "to",
    model_ready["week_start"].max(),
)
print(" grids:", model_ready["grid_id"].nunique())
print(" removed rows:", f"{len(panel_previous_weather) - len(model_ready):,}")

print("\nPrevious-week weather missing ratios:")
display(panel_previous_weather[LAG_WEATHER_COLS].isna().mean())

if event_col is not None:
    print("event rows in full panel       :", event_all_n)
    print("event rows in model-ready panel:", event_model_n)
    print("event retention ratio          :", event_retention)

# 週別カバレッジ
weekly_coverage = (
    panel_previous_weather
    .groupby("week_start")[LAG_WEATHER_COLS]
    .agg(lambda s: s.notna().mean())
    .reset_index()
)

# グリッド別カバレッジ
grid_coverage = (
    panel_previous_weather
    .groupby("grid_id")[LAG_WEATHER_COLS]
    .agg(lambda s: s.notna().mean())
    .reset_index()
)

display(weekly_coverage.head(10))
display(weekly_coverage.tail(10))
display(grid_coverage[LAG_WEATHER_COLS].describe())

## 9. 発生行の確認

発生フラグ列が存在する場合，前週気象欠測によってモデル用パネルから除外される発生行を保存する．

In [ ]:
if event_col is not None:
    event_mask = pd.to_numeric(
        panel_previous_weather[event_col],
        errors="coerce",
    ).fillna(0).eq(1)

    event_rows = panel_previous_weather.loc[event_mask].copy()
    event_rows["previous_weather_complete"] = event_rows[
        LAG_WEATHER_COLS
    ].notna().all(axis=1)

    event_check_cols = [
        c
        for c in [
            "grid_id",
            "week_start",
            "weather_source_week_start",
            event_col,
            "outbreak_count",
            "centroid_lon",
            "centroid_lat",
        ]
        + LAG_WEATHER_COLS
        + ["previous_weather_complete"]
        if c in event_rows.columns
    ]

    event_check = event_rows[event_check_cols].sort_values(
        ["week_start", "grid_id"]
    )

    save_csv(
        event_check,
        AUDIT_DIR / "04b_event_rows_previous_week_weather_check.csv",
        overwrite=OVERWRITE_OUTPUTS,
    )

    dropped_event_rows = event_check[
        ~event_check["previous_weather_complete"]
    ].copy()

    save_csv(
        dropped_event_rows,
        AUDIT_DIR / "04b_event_rows_dropped_for_missing_previous_weather.csv",
        overwrite=OVERWRITE_OUTPUTS,
    )

    print("all event rows    :", len(event_check))
    print("dropped event rows:", len(dropped_event_rows))
    display(event_check.head(100))
    if len(dropped_event_rows) > 0:
        display(dropped_event_rows)
else:
    event_check = pd.DataFrame()
    dropped_event_rows = pd.DataFrame()
    print(
        "発生フラグ列がないため，発生行単位の診断は省略します．"
    )

## 10. 出力ファイルの保存

主ファイルは`processed`直下，診断ファイルは
`processed/04b_previous_week_weather_audit`に保存する．

In [ ]:
# ============================================================
# 主出力
# ============================================================
previous_weather_path = (
    PROC_DIR / "weather_weekly_previous_week_features.parquet"
)
full_panel_path = (
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather.parquet"
)
model_ready_path = (
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet"
)

save_parquet(
    previous_weather,
    previous_weather_path,
    overwrite=OVERWRITE_OUTPUTS,
)
save_parquet(
    panel_previous_weather,
    full_panel_path,
    overwrite=OVERWRITE_OUTPUTS,
)
save_parquet(
    model_ready,
    model_ready_path,
    overwrite=OVERWRITE_OUTPUTS,
)

# 先頭サンプル
save_csv(
    previous_weather.head(SAMPLE_N),
    PROC_DIR / "weather_weekly_previous_week_features_sample.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
save_csv(
    panel_previous_weather.head(SAMPLE_N),
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_sample.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
save_csv(
    model_ready.head(SAMPLE_N),
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready_sample.csv",
    overwrite=OVERWRITE_OUTPUTS,
)

if SAVE_FULL_CSV:
    save_csv(
        panel_previous_weather,
        PROC_DIR
        / "hpai_weekly_grid_panel_with_previous_week_weather.csv",
        overwrite=OVERWRITE_OUTPUTS,
    )
    save_csv(
        model_ready,
        PROC_DIR
        / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.csv",
        overwrite=OVERWRITE_OUTPUTS,
    )

# ============================================================
# 診断出力
# ============================================================
save_csv(
    weekly_coverage,
    AUDIT_DIR / "04b_previous_week_weather_coverage_by_week.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
save_csv(
    grid_coverage,
    AUDIT_DIR / "04b_previous_week_weather_coverage_by_grid.csv",
    overwrite=OVERWRITE_OUTPUTS,
)

missing_by_week = weekly_coverage[
    weekly_coverage[LAG_WEATHER_COLS].min(axis=1) < 1.0
].copy()
save_csv(
    missing_by_week,
    AUDIT_DIR / "04b_weeks_with_incomplete_previous_weather.csv",
    overwrite=OVERWRITE_OUTPUTS,
)

schema_df = pd.DataFrame(
    {
        "column": panel_previous_weather.columns,
        "dtype": [
            str(panel_previous_weather[c].dtype)
            for c in panel_previous_weather.columns
        ],
        "missing_n": [
            int(panel_previous_weather[c].isna().sum())
            for c in panel_previous_weather.columns
        ],
        "missing_ratio": [
            float(panel_previous_weather[c].isna().mean())
            for c in panel_previous_weather.columns
        ],
    }
)
save_csv(
    schema_df,
    AUDIT_DIR / "04b_output_schema_and_missingness.csv",
    overwrite=OVERWRITE_OUTPUTS,
)

print("\nMain outputs saved.")

## 11. 最終検証と実行報告

重要な検査を一覧化し，1つでも不合格なら停止する．

In [ ]:
final_checks = pd.DataFrame(
    [
        {
            "check": "full panel row count equals input panel",
            "value": len(panel_previous_weather),
            "expected": len(panel),
            "passed": len(panel_previous_weather) == len(panel),
        },
        {
            "check": "full panel grid-week duplicates",
            "value": merged_duplicate_n,
            "expected": 0,
            "passed": merged_duplicate_n == 0,
        },
        {
            "check": "lag feature grid-week duplicates",
            "value": lag_duplicate_n,
            "expected": 0,
            "passed": lag_duplicate_n == 0,
        },
        {
            "check": "matched weather week gap is exactly 7 days",
            "value": (
                sorted(merged_gap_days.unique().tolist())
                if len(merged_gap_days) > 0
                else []
            ),
            "expected": [7],
            "passed": (
                len(merged_gap_days) > 0
                and merged_gap_days.eq(7).all()
            ),
        },
        {
            "check": "previous-week temperature order errors",
            "value": len(bad_temp_order),
            "expected": 0,
            "passed": len(bad_temp_order) == 0,
        },
        {
            "check": "model-ready period starts at fixed analysis start",
            "value": str(model_ready["week_start"].min().date()),
            "expected": str(ANALYSIS_START.date()),
            "passed": model_ready["week_start"].min() == ANALYSIS_START,
        },
        {
            "check": "model-ready period ends at fixed analysis end",
            "value": str(model_ready["week_start"].max().date()),
            "expected": str(ANALYSIS_END.date()),
            "passed": model_ready["week_start"].max() == ANALYSIS_END,
        },
        {
            "check": "model-ready weather missing values",
            "value": int(
                model_ready[LAG_WEATHER_COLS].isna().sum().sum()
            ),
            "expected": 0,
            "passed": (
                model_ready[LAG_WEATHER_COLS]
                .isna()
                .sum()
                .sum()
                == 0
            ),
        },
        {
            "check": "current-week weather columns absent from output",
            "value": [
                c
                for c in CURRENT_WEATHER_COLS
                if c in panel_previous_weather.columns
            ],
            "expected": [],
            "passed": not any(
                c in panel_previous_weather.columns
                for c in CURRENT_WEATHER_COLS
            ),
        },
        {
            "check": "lag values equal source-week values in sample",
            "value": int(value_check["mismatch_rows"].sum()),
            "expected": 0,
            "passed": bool(value_check["passed"].all()),
        },
    ]
)

save_csv(
    final_checks,
    AUDIT_DIR / "04b_final_validation_checks.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
display(final_checks)

if not final_checks["passed"].all():
    failed = final_checks[~final_checks["passed"]]
    display(failed)
    raise AssertionError(
        "最終検証に失敗しました．上のfailed checksを確認してください．"
    )

run_summary = {
    "notebook": "04b_build_previous_week_weather_panel_final.ipynb",
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "base_dir": str(BASE_DIR),
    "panel_input": str(panel_path),
    "weather_input": weather_source_description,
    "panel_input_rows": int(len(panel)),
    "panel_input_columns": int(panel.shape[1]),
    "output_full_rows": int(len(panel_previous_weather)),
    "output_full_columns": int(panel_previous_weather.shape[1]),
    "output_model_ready_rows": int(len(model_ready)),
    "output_model_ready_columns": int(model_ready.shape[1]),
    "grid_count_full": int(
        panel_previous_weather["grid_id"].nunique()
    ),
    "grid_count_model_ready": int(
        model_ready["grid_id"].nunique()
    ),
    "full_week_min": str(
        panel_previous_weather["week_start"].min().date()
    ),
    "full_week_max": str(
        panel_previous_weather["week_start"].max().date()
    ),
    "model_week_min": str(model_ready["week_start"].min().date()),
    "model_week_max": str(model_ready["week_start"].max().date()),
    "event_column": event_col,
    "event_rows_full": (
        int(event_all_n) if pd.notna(event_all_n) else None
    ),
    "event_rows_model_ready": (
        int(event_model_n) if pd.notna(event_model_n) else None
    ),
    "event_retention_ratio": (
        float(event_retention)
        if pd.notna(event_retention)
        else None
    ),
    "lag_weather_columns": LAG_WEATHER_COLS,
    "current_week_weather_columns_removed": existing_removed_cols,
    "all_final_checks_passed": bool(
        final_checks["passed"].all()
    ),
    "outputs": {
        "previous_weather_features": str(previous_weather_path),
        "full_panel": str(full_panel_path),
        "model_ready_panel": str(model_ready_path),
        "audit_dir": str(AUDIT_DIR),
    },
}

summary_json_path = AUDIT_DIR / "04b_run_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("Saved:", summary_json_path)
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

In [ ]:
report_text = f'''# 04b previous-week weather panel: execution report

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Purpose
The weather observed in week t-1 was shifted forward by seven days and used
as the predictor for HPAI risk in week t.

## Inputs
- Panel: `{panel_path}`
- Weekly weather: `{weather_source_description}`

## Fixed analysis period
- Analysis start: {ANALYSIS_START.date()}
- Analysis end: {ANALYSIS_END.date()}
- Rows after the analysis end were not treated as negative model observations.

## Output panel
- Full rows: {len(panel_previous_weather):,}
- Model-ready rows: {len(model_ready):,}
- Full grid count: {panel_previous_weather["grid_id"].nunique():,}
- Model-ready grid count: {model_ready["grid_id"].nunique():,}
- Full period: {panel_previous_weather["week_start"].min().date()} to {panel_previous_weather["week_start"].max().date()}
- Model-ready period: {model_ready["week_start"].min().date()} to {model_ready["week_start"].max().date()}

## Weather predictors
- `temp_mean_c_lag1w`
- `temp_min_c_lag1w`
- `temp_max_c_lag1w`

The source observation week is retained in `weather_source_week_start`.
For every matched row, `week_start - weather_source_week_start = 7 days`.

## Leakage control
Current-week weather columns were removed from the final output.
Only previous-week weather columns should be used by downstream models.

## Event retention
- Event column: {event_col}
- Event rows in full panel: {event_all_n}
- Event rows in model-ready panel: {event_model_n}
- Event retention ratio: {event_retention}

## Main outputs
- `{previous_weather_path}`
- `{full_panel_path}`
- `{model_ready_path}`

## Validation
All final validation checks passed: {bool(final_checks["passed"].all())}
'''

report_path = AUDIT_DIR / "04b_execution_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Saved:", report_path)
print(report_text)

## 12. 保存ファイルの存在確認

Drive上の主要出力と監査ファイルを確認する．

In [ ]:
expected_files = [
    previous_weather_path,
    full_panel_path,
    model_ready_path,
    PROC_DIR / "weather_weekly_previous_week_features_sample.csv",
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_sample.csv",
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready_sample.csv",
    AUDIT_DIR / "04b_input_file_log.csv",
    AUDIT_DIR / "04b_input_validation.csv",
    AUDIT_DIR / "04b_previous_week_value_verification.csv",
    AUDIT_DIR / "04b_previous_week_weather_coverage_by_week.csv",
    AUDIT_DIR / "04b_previous_week_weather_coverage_by_grid.csv",
    AUDIT_DIR / "04b_output_schema_and_missingness.csv",
    AUDIT_DIR / "04b_final_validation_checks.csv",
    AUDIT_DIR / "04b_run_summary.json",
    AUDIT_DIR / "04b_execution_report.md",
]

output_inventory_rows = []
for path in expected_files:
    path = Path(path)
    output_inventory_rows.append(
        {
            "file": path.name,
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": (
                int(path.stat().st_size) if path.exists() else 0
            ),
            "modified_at": (
                datetime.fromtimestamp(
                    path.stat().st_mtime
                ).isoformat(timespec="seconds")
                if path.exists()
                else None
            ),
        }
    )

output_inventory = pd.DataFrame(output_inventory_rows)
save_csv(
    output_inventory,
    AUDIT_DIR / "04b_output_file_inventory.csv",
    overwrite=OVERWRITE_OUTPUTS,
)
display(output_inventory)

missing_outputs = output_inventory[~output_inventory["exists"]]
if len(missing_outputs) > 0:
    display(missing_outputs)
    raise FileNotFoundError(
        "保存予定ファイルの一部が見つかりません．"
    )

print("All expected 04b outputs were saved successfully.")
print("\n次に使用するモデル入力:")
print(model_ready_path)

## 次の工程

次のモデル比較ノートブックでは，入力ファイルを次に固定する．

```text
processed/hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet
```

モデル特徴量には次の3列を使用する．

```text
temp_mean_c_lag1w
temp_min_c_lag1w
temp_max_c_lag1w
```

以下の現在週気象列は使用しない．

```text
temp_mean_c
temp_min_c
temp_max_c
```